In [2]:
import os

# Define the exact directory layout from your class
dirs = [
    "rag-agent/app/rag",
    "rag-agent/data/uploads",
    "rag-agent/data/faiss_index",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Structure created.")


Structure created.


In [3]:
%%writefile rag-agent/requirements.txt
fastapi==0.115.0
uvicorn[standard]==0.30.6
python-multipart==0.0.9
groq==0.11.0
faiss-cpu
fastembed==0.3.6
pypdf==5.0.1
python-dotenv==1.0.1
pydantic==2.9.2
numpy==1.26.4


Writing rag-agent/requirements.txt


In [4]:
%%writefile rag-agent/Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT:-8000}"]


Writing rag-agent/Dockerfile


In [5]:
%%writefile rag-agent/.dockerignore
__pycache__/
*.pyc
.env
.git
.gitignore
*.md
.ipynb_checkpoints
data/uploads/*
data/faiss_index/*
!data/uploads/.gitkeep
!data/faiss_index/.gitkeep


Writing rag-agent/.dockerignore


In [6]:
%%writefile rag-agent/app/__init__.py
# Package initializer for app


Writing rag-agent/app/__init__.py


In [7]:
%%writefile rag-agent/app/rag/__init__.py
# Package initializer for rag logic


Writing rag-agent/app/rag/__init__.py


In [8]:
%%writefile rag-agent/app/config.py
import os
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

# fastembed model, ONNX-runtime based (no PyTorch) -> low RAM footprint.
# BAAI/bge-small-en-v1.5: 384-dim, ~130MB on disk, strong quality for its size.
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "800"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "120"))

TOP_K = int(os.getenv("TOP_K", "4"))

DATA_DIR = BASE_DIR / "data"
UPLOAD_DIR = DATA_DIR / "uploads"
INDEX_DIR = DATA_DIR / "faiss_index"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY is not set. Add it as an environment variable "
        "(Railway: Project -> Variables -> GROQ_API_KEY)."
    )


Writing rag-agent/app/config.py


In [9]:
%%writefile rag-agent/app/system_prompt.py
SYSTEM_PROMPT = """
You are an Agent on a Digital Skills Website, what you do is to answer enquiries and questions about the Business Analytics with AI, using the information provided.

Always respond warmly, if you are unable to answer, refer the user to Olumatin Thomas on 07037613488
"""


Writing rag-agent/app/system_prompt.py


In [10]:
%%writefile rag-agent/app/models.py
from typing import List, Optional
from pydantic import BaseModel

class ChatRequest(BaseModel):
    question: str
    top_k: Optional[int] = None

class SourceChunk(BaseModel):
    text: str
    source: str
    chunk_id: int
    score: float

class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceChunk]

class UploadResponse(BaseModel):
    filename: str
    chunks_added: int
    total_chunks: int


Writing rag-agent/app/models.py


In [11]:
%%writefile rag-agent/app/rag/ingest.py
import pickle
import numpy as np
import faiss
from pypdf import PdfReader
from fastembed import TextEmbedding

from app import config

_embedder = None
_index = None
_metadata = []  # list of {"text":..., "source":..., "chunk_id":...}

INDEX_FILE = config.INDEX_DIR / "index.faiss"
META_FILE = config.INDEX_DIR / "metadata.pkl"


def get_embedder():
    """fastembed loads an ONNX model — no PyTorch, small RAM footprint."""
    global _embedder
    if _embedder is None:
        _embedder = TextEmbedding(model_name=config.EMBEDDING_MODEL)
    return _embedder


def _embed(texts):
    embedder = get_embedder()
    vecs = np.array(list(embedder.embed(texts)), dtype="float32")
    # normalize so FAISS IndexFlatIP behaves like cosine similarity
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    return vecs / norms


def load_index():
    """Load a persisted FAISS index from disk, or start a fresh empty one."""
    global _index, _metadata
    if INDEX_FILE.exists() and META_FILE.exists():
        _index = faiss.read_index(str(INDEX_FILE))
        with open(META_FILE, "rb") as f:
            _metadata = pickle.load(f)
    else:
        dim = _embed(["dimension probe"]).shape[1]
        _index = faiss.IndexFlatIP(dim)
        _metadata = []
    return _index, _metadata


def save_index():
    faiss.write_index(_index, str(INDEX_FILE))
    with open(META_FILE, "wb") as f:
        pickle.dump(_metadata, f)


def chunk_text(text, chunk_size=None, overlap=None):
    chunk_size = chunk_size or config.CHUNK_SIZE
    overlap = overlap or config.CHUNK_OVERLAP
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]


def extract_pdf_text(path):
    reader = PdfReader(str(path))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def ingest_pdf(path, source_name):
    """Extract, chunk, embed and add a PDF's contents to the FAISS index."""
    global _index, _metadata
    if _index is None:
        load_index()

    text = extract_pdf_text(path)
    chunks = chunk_text(text)
    if not chunks:
        return 0, len(_metadata)

    vectors = _embed(chunks)
    _index.add(vectors)

    start_id = len(_metadata)
    for i, chunk in enumerate(chunks):
        _metadata.append({
            "text": chunk,
            "source": source_name,
            "chunk_id": start_id + i,
        })

    save_index()
    return len(chunks), len(_metadata)


Writing rag-agent/app/rag/ingest.py


In [12]:
%%writefile rag-agent/app/rag/retriever.py
from app import config
from app.rag import ingest


def search(query, top_k=None):
    top_k = top_k or config.TOP_K

    if ingest._index is None or ingest._index.ntotal == 0:
        return []

    query_vec = ingest._embed([query])
    scores, indices = ingest._index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = ingest._metadata[idx]
        results.append({
            "text": meta["text"],
            "source": meta["source"],
            "chunk_id": meta["chunk_id"],
            "score": float(score),
        })
    return results


Writing rag-agent/app/rag/retriever.py


In [13]:
%%writefile rag-agent/app/rag/llm.py
from groq import Groq

from app import config
from app.system_prompt import SYSTEM_PROMPT

_client = None


def get_client():
    global _client
    if _client is None:
        _client = Groq(api_key=config.GROQ_API_KEY)
    return _client


def build_context_block(chunks):
    if not chunks:
        return "No relevant context was found in the knowledge base."
    parts = []
    for c in chunks:
        parts.append(f"[Source: {c['source']} | chunk {c['chunk_id']}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)


def generate_answer(question, chunks):
    context = build_context_block(chunks)
    client = get_client()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

    completion = client.chat.completions.create(
        model=config.GROQ_MODEL,
        messages=messages,
        temperature=0.2,
    )
    return completion.choices[0].message.content


Writing rag-agent/app/rag/llm.py


In [14]:
%%writefile rag-agent/app/main.py
import shutil
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import HTMLResponse

from app import config
from app.models import ChatRequest, ChatResponse, UploadResponse, SourceChunk
from app.rag import ingest, retriever, llm

app = FastAPI(title="RAG Agent Dashboard")


@app.on_event("startup")
def startup():
    ingest.load_index()


@app.get("/health")
def health():
    total = 0 if ingest._index is None else ingest._index.ntotal
    return {"status": "ok", "chunks_indexed": total}


@app.post("/upload", response_model=UploadResponse)
async def upload_pdf(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are supported.")

    dest = config.UPLOAD_DIR / file.filename
    with open(dest, "wb") as f:
        shutil.copyfileobj(file.file, f)

    added, total = ingest.ingest_pdf(dest, file.filename)
    return UploadResponse(filename=file.filename, chunks_added=added, total_chunks=total)


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    chunks = retriever.search(req.question, req.top_k)
    answer = llm.generate_answer(req.question, chunks)
    sources = [SourceChunk(**c) for c in chunks]
    return ChatResponse(answer=answer, sources=sources)


@app.get("/", response_class=HTMLResponse)
def serve_ui():
    return """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>RAG Agent Chat Dashboard</title>
        <link href="https://jsdelivr.net" rel="stylesheet">
        <style>
            body { background-color: #f4f6f9; font-family: 'Segoe UI', system-ui, sans-serif; }
            .chat-container { height: 450px; overflow-y: auto; background: white; border-radius: 10px; padding: 20px; box-shadow: inset 0 0 10px rgba(0,0,0,0.05); }
            .user-msg { background-color: #0d6efd; color: white; border-radius: 15px 15px 0 15px; padding: 10px 15px; margin: 5px 0; max-width: 75%; float: right; clear: both; }
            .agent-msg { background-color: #e9ecef; color: #212529; border-radius: 15px 15px 15px 0; padding: 10px 15px; margin: 5px 0; max-width: 75%; float: left; clear: both; }
            .sources-box { font-size: 0.8rem; color: #6c757d; margin-top: 5px; border-left: 2px solid #dee2e6; padding-left: 8px; }
            .card { border: none; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-radius: 12px; }
        </style>
    </head>
    <body>
        <div class="container py-5">
            <header class="pb-3 mb-4 border-bottom">
                <span class="fs-4 fw-bold text-dark">📚 Business Analytics RAG Assistant</span>
            </header>

            <div class="row g-4">
                <div class="col-md-4">
                    <div class="card p-4 bg-white mb-4">
                        <h5 class="fw-bold mb-3">📁 Upload Knowledge Base</h5>
                        <p class="text-muted small">Upload your course PDFs or business files here.</p>
                        <div class="mb-3">
                            <input class="form-control" type="file" id="pdfFile" accept=".pdf">
                        </div>
                        <button class="btn btn-dark w-100" onclick="uploadDocument()">Upload Document</button>
                        <div id="uploadStatus" class="mt-3 small"></div>
                    </div>
                </div>

                <div class="col-md-8">
                    <div class="card p-4 bg-white">
                        <h5 class="fw-bold mb-3">💬 Chat with Agent</h5>
                        <div class="chat-container border mb-3" id="chatWindow">
                            <div class="agent-msg">Hello! Ask me any questions about our Business Analytics with AI track.</div>
                        </div>
                        <div class="input-group">
                            <input type="text" id="userInput" class="form-control" placeholder="Type your inquiry here..." onkeypress="if(event.key === 'Enter') sendMessage()">
                            <button class="btn btn-primary" onclick="sendMessage()">Send</button>
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <script>
            async function uploadDocument() {
                const fileInput = document.getElementById('pdfFile');
                const statusDiv = document.getElementById('uploadStatus');
                if (!fileInput.files[0]) { statusDiv.innerHTML = '<span class="text-danger">Select a file first!</span>'; return; }

                const formData = new FormData();
                formData.append('file', fileInput.files[0]);
                statusDiv.innerHTML = '<div class="spinner-border spinner-border-sm text-primary"></div> Processing...';

                try {
                    const response = await fetch('/upload', { method: 'POST', body: formData });
                    const resData = await response.json();
                    if(response.ok) {
                        statusDiv.innerHTML = `<span class="text-success">Processed ${resData.filename} (${resData.chunks_added} chunks added).</span>`;
                    } else {
                        statusDiv.innerHTML = `<span class="text-danger">Error: ${resData.detail}</span>`;
                    }
                } catch(e) { statusDiv.innerHTML = '<span class="text-danger">Upload failed.</span>'; }
            }

            async function sendMessage() {
                const input = document.getElementById('userInput');
                const chatWindow = document.getElementById('chatWindow');
                const query = input.value.trim();
                if (!query) return;

                chatWindow.innerHTML += `<div class="user-msg">${query}</div>`;
                input.value = '';
                chatWindow.scrollTop = chatWindow.scrollHeight;

                try {
                    const response = await fetch('/chat', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ question: query })
                    });
                    const resData = await response.json();

                    let sourcesHtml = '';
                    if(resData.sources && resData.sources.length > 0) {
                        sourcesHtml = '<div class="sources-box"><strong>Sources:</strong> ' +
                            resData.sources.map(s => `${s.source} (id: ${s.chunk_id})`).join(', ') + '</div>';
                    }

                    chatWindow.innerHTML += `<div class="agent-msg"><div>${resData.answer}</div>${sourcesHtml}</div>`;
                } catch(e) {
                    chatWindow.innerHTML += `<div class="agent-msg text-danger">Failed to fetch answer.</div>`;
                }
                chatWindow.scrollTop = chatWindow.scrollHeight;
            }
        </script>
    </body>
    </html>
    """


Writing rag-agent/app/main.py


In [15]:
%%writefile rag-agent/data/uploads/.gitkeep
# Keeps uploads directory structure tracked


Writing rag-agent/data/uploads/.gitkeep


In [16]:
%%writefile rag-agent/data/faiss_index/.gitkeep
# Keeps database directory structure tracked


Writing rag-agent/data/faiss_index/.gitkeep


In [17]:
import shutil
from google.colab import files

# Create a zip package containing the entire rag-agent folder tree
shutil.make_archive("rag_agent_deployment", 'zip', "rag-agent")

print("Zip archive created successfully!")
# Trigger browser file download utility
files.download("rag_agent_deployment.zip")


Zip archive created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>